In [3]:
import requests, bs4, lxml, pandas
print("All libraries successfully imported!")

All libraries successfully imported!


In [2]:
import requests

url = "https://towardsdatascience.com/" # Step 2: Define a User-Agent header – essential for ethical and often successful scraping
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
# Step 3: Send the GET request with the specified headers
response = requests.get(url, headers=headers)
# Step 4: Verify the server's response status. A 200 code is a success!
print("--- Initiating Contact ---")
print(f"Status Code: {response.status_code}")
# Step 5: Get a peek at the raw HTML received.
print("HTML Content Preview (First 500 characters):")
print(response.text[:500])
print("---")

--- Initiating Contact ---
Status Code: 200
HTML Content Preview (First 500 characters):
<!DOCTYPE html>
<html lang="en-US">
<head>
	<meta charset="UTF-8" />
	<script src="https://h030.towardsdatascience.com/script.js"></script><!-- Google Tag Manager -->
<script>
	(function (w, d, s, l, i) {
		w[l] = w[l] || [];
		w[l].push({
			'gtm.start': new Date().getTime(),
			event: 'gtm.js'
		});
		var f = d.getElementsByTagName(s)[0],
			j = d.createElement(s),
			dl = l != 'dataLayer' ? '&l=' + l : '';
		j.async = true;
		j.src =
			'https://www.googletagmanager.com/gtm.js?id=' + i + dl;

---


In [5]:
import requests
from bs4 import BeautifulSoup

# Step 1: Ensure you have HTML content from a previous successful request
url = "https://towardsdatascience.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
response = requests.get(url, headers=headers)
response.raise_for_status()
# Crucial for good error handling
html_content = response.text

# Step 2: Transform the raw HTML string into a Beautiful Soup object using 'lxml' parser
soup = BeautifulSoup(html_content, 'lxml')
# Using 'lxml' for potentially faster parsing

# Step 3: Confirm successful parsing by accessing a basic element like the page title
print("--- Parsing HTML Content ---")
if soup.title:
    print(f"Parsed Page Title: {soup.title.text.strip()}")
else:
    print("Page title not found. HTML parsing might need adjustment.")
print("---")

--- Parsing HTML Content ---
Parsed Page Title: Towards Data Science
---


In [7]:
#TRY IT TASK 4
import requests
from bs4 import BeautifulSoup

# Step 1: Ensure you have HTML content from a previous successful request
url = "https://towardsdatascience.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
response = requests.get(url, headers=headers)
response.raise_for_status()
# Crucial for good error handling
html_content = response.text

# Step 2: Transform the raw HTML string into a Beautiful Soup object using 'lxml' parser
soup = BeautifulSoup(html_content, 'lxml')
# Using 'lxml' for potentially faster parsing

# Step 3: Confirm successful parsing by accessing a basic element like the page title
print("--- Parsing HTML Content ---")
if soup.title:
    print(f"Parsed Page Title: {soup.title.text.strip()}")
else:
    print("Page title not found. HTML parsing might need adjustment.")
print("---")

--- Parsing HTML Content ---
Parsed Page Title: Towards Data Science
---


In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')
    article_links_data = []

    # --- Strategy for Wikipedia's Main Page ---
    # We'll look for links within specific sections like "In the news" or "On this day".
    # These sections are often within `div`s or `td`s with specific IDs or classes.

    # 1. Target the "In the news" section
    # The 'In the news' section is typically within a div with id 'mp-itn'
    in_the_news_div = soup.find('div', id='mp-itn')
    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        # Articles are usually direct links within <li> or <p> tags
        # Find all <a> tags within this div
        links = in_the_news_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            # Filter out non-article links (e.g., [edit], citations, external links not starting with /wiki/)
            if title_text and href.startswith('/wiki/') and ':' not in href:
                # Avoid special pages like 'Help:', 'Category:'
                absolute_url = urljoin(url, href)
                # Ensure it's not a duplicate if this logic is refined later
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })
                    # Add to list temporarily, then process for unique later if combining sections

    # 2. Target the "On this day" section
    # The 'On this day' section is typically within a div with id 'mp-otd'
    on_this_day_div = soup.find('div', id='mp-otd')
    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    # Avoid adding duplicates if combining sources
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    # Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10 from all sections):")
        # Take up to the first 10 unique articles found
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL: {article['url']}")
    else:
        print("\nNo article links found in the targeted sections ('In the news', 'On this day').")
        print("The HTML structure might have changed, or the selectors need adjustment.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10 from all sections):
- Title: 'Antigua and Barbuda Labour Party'
  URL: https://en.wikipedia.org/wiki/Antigua_and_Barbuda_Labour_Party
- Title: 'Prime Minister'
  URL: https://en.wikipedia.org/wiki/Prime_Minister_of_Antigua_and_Barbuda
- Title: 'Gaston Browne'
  URL: https://en.wikipedia.org/wiki/Gaston_Browne
- Title: 'the Antiguan general election'
  URL: https://en.wikipedia.org/wiki/2026_Antiguan_general_election
- Title: 'J. Craig Venter'
  URL: https://en.wikipedia.org/wiki/J._Craig_Venter
- Title: 'sequencing of the human genome'
  URL: https://en.wikipedia.org/wiki/Human_Genome_Project#Public_versus_private_approaches
- Title: 'A train crash'
  URL: https://en.wikipedia.org/wiki/2026_Bekasi_train_crash
- Title: 'Jakarta'
  URL: https://en.wikipedia.org/wiki/Jakarta
- Title: 'the London Marathon'
  URL: 

In [10]:
#TRY IT TASK 5
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

print(f"Attempting to scrape: {url}")

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    article_links_data = []

    # --- Extract Main Title ---
    main_title = soup.find('h1')
    if main_title:
        print(f"\n--- Main Title ---")
        print(f"Title: {main_title.text.strip()}")

    # --- Extract Introductory Paragraph ---
    intro_para = soup.find('p')
    if intro_para:
        print(f"\n--- Introductory Paragraph ---")
        print(f"Content: {intro_para.text.strip()[:200]}...")  # First 200 characters

    # --- Extract Last Modified Date ---
    last_modified = soup.find('li', id='footer-info-lastmod')
    if last_modified:
        print(f"\n--- Last Modified ---")
        print(f"Info: {last_modified.text.strip()}")

    # --- Extract Featured Article Title ---
    featured_div = soup.find('div', id='mp-featured')
    if featured_div:
        print(f"\n--- Featured Article ---")
        featured_title = featured_div.find('a', href=True)
        if featured_title:
            print(f"Title: {featured_title.text.strip()}")
            print(f"URL: {urljoin(url, featured_title['href'])}")

    # --- Original: "In the news" section ---
    in_the_news_div = soup.find('div', id='mp-itn')
    if in_the_news_div:
        print("\n--- From 'In the news' section ---")
        links = in_the_news_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    # --- Original: "On this day" section ---
    on_this_day_div = soup.find('div', id='mp-otd')
    if on_this_day_div:
        print("\n--- From 'On this day' section ---")
        links = on_this_day_div.find_all('a', href=True)
        for link in links:
            title_text = link.text.strip()
            href = link['href']
            if title_text and href.startswith('/wiki/') and ':' not in href:
                absolute_url = urljoin(url, href)
                if not any(data['url'] == absolute_url for data in article_links_data):
                    article_links_data.append({
                        'title': title_text,
                        'url': absolute_url
                    })

    # Display results
    if article_links_data:
        print("\nExtracted Article Titles and URLs (First 10 from all sections):")
        for i, article in enumerate(article_links_data[:10]):
            print(f"- Title: '{article['title']}'")
            print(f"  URL: {article['url']}")
    else:
        print("\nNo article links found in the targeted sections ('In the news', 'On this day').")
        print("The HTML structure might have changed, or the selectors need adjustment.")

except requests.exceptions.RequestException as e:
    print(f"\nError fetching URL {url}: {e}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

print("---")

Attempting to scrape: https://en.wikipedia.org/wiki/Main_Page

--- Main Title ---
Title: Main Page

--- Introductory Paragraph ---
Content: Maurice Suckling (4 May 1726 – 14 July 1778) was a Royal Navy officer and politician. He saw service in the English Channel and Mediterranean Sea during the War of the Austrian Succession. At the star...

--- Last Modified ---
Info: This page was last edited on 25 March 2026, at 16:10 (UTC).

--- From 'In the news' section ---

--- From 'On this day' section ---

Extracted Article Titles and URLs (First 10 from all sections):
- Title: 'Antigua and Barbuda Labour Party'
  URL: https://en.wikipedia.org/wiki/Antigua_and_Barbuda_Labour_Party
- Title: 'Prime Minister'
  URL: https://en.wikipedia.org/wiki/Prime_Minister_of_Antigua_and_Barbuda
- Title: 'Gaston Browne'
  URL: https://en.wikipedia.org/wiki/Gaston_Browne
- Title: 'the Antiguan general election'
  URL: https://en.wikipedia.org/wiki/2026_Antiguan_general_election
- Title: 'J. Craig Venter'
  U

In [ ]:
#TRY IT TASK 6
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://en.wikipedia.org/wiki/List_of_datasets_for_machine_learning_research"
# A relevant table
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')
# Using 'lxml' parser

print("--- PROCEDURE 6: Extract Data from HTML Tables ---")

# Step 1: Identify the specific table you want to extract
target_table = soup.find('table', class_='wikitable')
if target_table:
    # Step 2: Extract table headers (column names)
    table_headers = [th.text.strip() for th in target_table.find('tr').find_all('th')]
    print(f"Table Headers (First 5): {table_headers[:5]}")

    # Step 3: Extract data rows, skipping the header row
    table_data = []
    for row in target_table.find_all('tr')[1:6]:
        # Limiting to first 5 data rows for demo
        cells = row.find_all(['td', 'th'])
        # Get both data cells and potential row headers
        cell_texts = [cell.text.strip() for cell in cells]
        table_data.append(cell_texts)

    print("\nFirst 5 Data Rows (Raw Lists):")
    for i, row in enumerate(table_data):
        print(f"Row {i+1}: {row[:5]}...")
        # Display snippet

    # Step 4: (Optional but Recommended) Convert the data into a Pandas DataFrame
    if table_data:
        try:
            # Adjust columns to match actual data rows length in case of merged cells etc.
            df = pd.DataFrame(table_data, columns=table_headers[:len(table_data[0])])
            print("\nDataFrame Representation (Head of first 3 rows):")
            print(df.head(3).to_string())
        except ValueError as e:
            print(f"\nCould not create DataFrame. Column count mismatch: {e}")
    else:
        print("No data rows found in the table.")
else:
    print("Target table with class 'wikitable' not found on the page.")

print("---")

--- PROCEDURE 6: Extract Data from HTML Tables ---
Table Headers (First 5): ['Type', 'Subtypes']

First 5 Data Rows (Raw Lists):
Row 1: ['Specific category', 'Finance, Economics, Commerce, Societal, Health, Academy, Sports, Food, Agriculture, Travel, Geospatial, Political, Consumer, Transport,  Logistics, Environmental, Real-Estate, Legal, Entertainment, Energy, Hospitality']...
Row 2: ['Scope', 'Supranational Union, National, Subnational, Municipality, Urban, Rural']...
Row 3: ['Language', 'Mandarin Chinese, Spanish, English, Arabic, Hindi, Bengali']...
Row 4: ['Type', 'Tabular, Graph, Text, Image, Sound, Video']...
Row 5: ['Usage', 'Training, validating, and testing']...

DataFrame Representation (Head of first 3 rows):
                Type                                                                                                                                                                                                                   Subtypes
0  Specific category  Financ

In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://en.wikipedia.org/wiki/Comparison_of_programming_languages"
# Wikipedia page with programming language comparison tables
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')
# Using 'lxml' parser

print("--- PROCEDURE 6: Extract Data from Programming Languages Comparison Table ---")

# Step 1: Find all wikitable classes (there may be multiple tables on this page)
tables = soup.find_all('table', class_='wikitable')
print(f"Found {len(tables)} table(s) on this page.\n")

# Step 2: Extract from the first relevant table (usually the main comparison table)
if tables:
    target_table = tables[0]
    # Step 3: Extract table headers (column names)
    table_headers = [th.text.strip() for th in target_table.find('tr').find_all('th')]
    print(f"Table Headers: {table_headers}")

    # Step 4: Extract data rows, skipping the header row
    table_data = []
    for row in target_table.find_all('tr')[1:11]:
        # Limiting to first 10 data rows for demo
        cells = row.find_all(['td', 'th'])
        # Get both data cells and potential row headers
        cell_texts = [cell.text.strip() for cell in cells]
        if cell_texts:
            # Only add non-empty rows
            table_data.append(cell_texts)

    print(f"\nExtracted {len(table_data)} data rows.\n")
    print("First 5 Data Rows (Raw Lists):")
    for i, row in enumerate(table_data[:5]):
        print(f"Row {i+1}: {row[:6]}...")
        # Display snippet (first 6 columns)

    # Step 5: Convert the data into a Pandas DataFrame
    if table_data:
        try:
            # Adjust columns to match actual data rows length
            df = pd.DataFrame(table_data, columns=table_headers[:len(table_data[0])])
            print("\nDataFrame Representation (First 5 rows):")
            print(df.head(5).to_string())
            print(f"\nDataFrame Shape: {df.shape}")
            print(f"Total Columns: {len(df.columns)}")
            print(f"Total Rows: {len(df)}")
        except ValueError as e:
            print(f"\nCould not create DataFrame. Column count mismatch: {e}")
            print(f"Header count: {len(table_headers)}, Data row column count: {len(table_data[0])}")
    else:
        print("No data rows found in the table.")
else:
    print("No wikitable found on the page.")

print("---")

--- PROCEDURE 6: Extract Data from Programming Languages Comparison Table ---
Found 2 table(s) on this page.

Table Headers: ['Language', 'Original purpose', 'Imperative', 'Object-oriented', 'Functional', 'Procedural', 'Generic', 'Reflective', 'Other paradigms', 'Standardized']

Extracted 10 data rows.

First 5 Data Rows (Raw Lists):
Row 1: ['1C:Enterprise programming language', 'Application, RAD, business, general, web, mobile', 'Yes', 'No', 'Yes', 'Yes']...
Row 2: ['ActionScript', 'Application, client-side, web', 'Yes', 'Yes', 'Yes', 'Yes']...
Row 3: ['Ada', 'Application, embedded, realtime, system', 'Yes', 'Yes[2]', 'No', 'Yes[3]']...
Row 4: ['Aldor', 'Highly domain-specific, symbolic computing', 'Yes', 'Yes', 'Yes', 'No']...
Row 5: ['ALGOL 58', 'Application', 'Yes', 'No', 'No', 'No']...

DataFrame Representation (First 5 rows):
                             Language                                  Original purpose Imperative Object-oriented Functional Procedural Generic Reflective 

In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---")

start_url = "http://quotes.toscrape.com/"
base_domain = urlparse(start_url).netloc
all_quotes = []
# We'll collect the actual quotes here
pages_to_collect = 3
# Limiting to 3 pages for demonstration
current_page_url = start_url
page_counter = 0

while page_counter < pages_to_collect:
    print(f"\nAttempting to fetch page {page_counter + 1}: {current_page_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status()
        # Check for HTTP errors (e.g., 404, 500)
        soup = BeautifulSoup(response.text, 'lxml')
        # Using 'lxml' parser

        # Step 1: Extract data from the current page (Quotes)
        # On Quotes to Scrape, each quote is in a <div> with class 'quote'
        quote_elements = soup.find_all('div', class_='quote')
        if quote_elements:
            for quote_item in quote_elements:
                text = quote_item.find('span', class_='text').text.strip()
                author = quote_item.find('small', class_='author').text.strip()
                all_quotes.append({'text': text, 'author': author})
            print(f"Collected {len(quote_elements)} quotes from this page.")
        else:
            print("No quote elements found on this page. Stopping.")
            break
            # No quotes means no more content

        # Step 2: Locate the "Next Page" link
        # On Quotes to Scrape, the 'next' button is usually a <li> with class 'next'
        # inside a <ul> element.
        next_li = soup.find('li', class_='next')
        if next_li:
            next_link_element = next_li.find('a', href=True)
            if next_link_element:
                next_page_relative_url = next_link_element['href']
                current_page_url = urljoin(start_url, next_page_relative_url)
                page_counter += 1
                time.sleep(1)
                # Be polite: pause before fetching the next page
            else:
                print("Next page link tag not found within 'next' li. Stopping.")
                break
        else:
            print("No 'next' pagination link found. Assuming last page.")
            break
            # Exit loop if no next page link found

    except requests.exceptions.RequestException as e:
        print(f"ERROR (Network/HTTP): Could not fetch {current_page_url}: {e}")
        break
        # Stop on error
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {current_page_url}: {e}")
        break

print(f"\n--- Total Quotes Collected Across {page_counter} pages: {len(all_quotes)} ---")
print("First 5 Collected Quotes (Sample):")
for i, quote in enumerate(all_quotes[:5]):
    print(f"- \"{quote['text']}\" - {quote['author']}")
print("--- End of Procedure ---")

--- PROCEDURE 7: Handle Pagination (Quotes to Scrape) ---

Attempting to fetch page 1: http://quotes.toscrape.com/
Collected 10 quotes from this page.

Attempting to fetch page 2: http://quotes.toscrape.com/page/2/
Collected 10 quotes from this page.

Attempting to fetch page 3: http://quotes.toscrape.com/page/3/
Collected 10 quotes from this page.

--- Total Quotes Collected Across 3 pages: 30 ---
First 5 Collected Quotes (Sample):
- "“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”" - Albert Einstein
- "“It is our choices, Harry, that show what we truly are, far more than our abilities.”" - J.K. Rowling
- "“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”" - Albert Einstein
- "“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”" - Jane Austen
- "“Imperfection is beauty, madness is geni

In [ ]:
#TRY IT TASK 7 - Revised for Books to Scrape
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

print("--- PROCEDURE 7: Handle Pagination (Books to Scrape) ---")

start_url = "http://books.toscrape.com/"
base_domain = urlparse(start_url).netloc
all_books = []
# We'll collect the actual books here
pages_to_collect = 3
# Limiting to 3 pages for demonstration
current_page_url = start_url
page_counter = 0

while page_counter < pages_to_collect:
    print(f"\nAttempting to fetch page {page_counter + 1}: {current_page_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(current_page_url, headers=headers, timeout=10)
        response.raise_for_status()
        # Check for HTTP errors (e.g., 404, 500)
        soup = BeautifulSoup(response.text, 'lxml')
        # Using 'lxml' parser

        # Step 1: Extract data from the current page (Books)
        # On Books to Scrape, each book is in an <article> with class 'product_pod'
        book_elements = soup.find_all('article', class_='product_pod')
        if book_elements:
            for book_item in book_elements:
                # Extract book title from <h3> > <a> tag
                title_element = book_item.find('h3').find('a')
                title = title_element.get('title', 'N/A').strip()
                
                # Extract book URL
                book_url = title_element.get('href', 'N/A')
                book_url = urljoin(current_page_url, book_url)
                
                # Extract price from <p> with class 'price_color'
                price_element = book_item.find('p', class_='price_color')
                price = price_element.text.strip() if price_element else 'N/A'
                
                # Extract availability from <p> with class 'instock availability'
                availability_element = book_item.find('p', class_='instock availability')
                availability = availability_element.text.strip() if availability_element else 'N/A'
                
                # Extract rating from <p> with class 'star-rating'
                rating_element = book_item.find('p', class_='star-rating')
                rating = rating_element.get('class')[1] if rating_element else 'N/A'
                
                all_books.append({
                    'title': title,
                    'url': book_url,
                    'price': price,
                    'availability': availability,
                    'rating': rating
                })
            print(f"Collected {len(book_elements)} books from this page.")
        else:
            print("No book elements found on this page. Stopping.")
            break
            # No books means no more content

        # Step 2: Locate the "Next Page" link
        # On Books to Scrape, the 'next' button is a <li> with class 'next'
        # containing an <a> tag with href pointing to the next page
        next_li = soup.find('li', class_='next')
        if next_li:
            next_link_element = next_li.find('a', href=True)
            if next_link_element:
                next_page_relative_url = next_link_element['href']
                current_page_url = urljoin(current_page_url, next_page_relative_url)
                page_counter += 1
                time.sleep(1)
                # Be polite: pause before fetching the next page
            else:
                print("Next page link tag not found within 'next' li. Stopping.")
                break
        else:
            print("No 'next' pagination link found. Assuming last page.")
            break
            # Exit loop if no next page link found

    except requests.exceptions.RequestException as e:
        print(f"ERROR (Network/HTTP): Could not fetch {current_page_url}: {e}")
        break
        # Stop on error
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {current_page_url}: {e}")
        break

print(f"\n--- Total Books Collected Across {page_counter} pages: {len(all_books)} ---")
print("\nFirst 5 Collected Books (Sample):")
for i, book in enumerate(all_books[:5]):
    print(f"\n{i+1}. Title: {book['title']}")
    print(f"   Price: {book['price']}")
    print(f"   Rating: {book['rating']}")
    print(f"   Availability: {book['availability']}")
    print(f"   URL: {book['url']}")
print("\n--- End of Procedure ---")

--- PROCEDURE 7: Handle Pagination (Books to Scrape) ---

Attempting to fetch page 1: http://books.toscrape.com/
Collected 20 books from this page.

Attempting to fetch page 2: http://books.toscrape.com/catalogue/page-2.html
Collected 20 books from this page.

Attempting to fetch page 3: http://books.toscrape.com/catalogue/page-3.html
Collected 20 books from this page.

--- Total Books Collected Across 3 pages: 60 ---

First 5 Collected Books (Sample):

1. Title: A Light in the Attic
   Price: Â£51.77
   Rating: Three
   Availability: In stock
   URL: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html

2. Title: Tipping the Velvet
   Price: Â£53.74
   Rating: One
   Availability: In stock
   URL: http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html

3. Title: Soumission
   Price: Â£50.10
   Rating: One
   Availability: In stock
   URL: http://books.toscrape.com/catalogue/soumission_998/index.html

4. Title: Sharp Objects
   Price: Â£47.82
   Ratin

In [20]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

print("--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---")

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]

        # Define the exact headers we now expect based on your output
        # We'll use 'Creator' and 'Initial release' since 'Latest stable release version' is absent
        software_idx = headers_text.index('Software') if 'Software' in headers_text else -1
        creator_idx = headers_text.index('Creator') if 'Creator' in headers_text else -1
        initial_release_idx = headers_text.index('Initial release') if 'Initial release' in headers_text else -1

        # Check if all required headers were found
        required_indices = [software_idx, creator_idx, initial_release_idx]
        required_names = ['Software', 'Creator', 'Initial release']

        if any(idx == -1 for idx in required_indices):
            print("ERROR: One or more required columns are still missing in the table headers based on current revision.")
            print(f"Expected: {required_names}")
            print("Found headers:", headers_text)
            exit()
            # Terminate if critical columns are not present

        cleaned_data_records = []
        data_rows = table.find_all('tr')[1:]
        # Skip header row

        # Process all rows in the table
        for row_num, row in enumerate(data_rows):
            cols = row.find_all('td')
            # Ensure row has enough columns for the indices we're using
            if len(cols) > max(required_indices):
                software_name = cols[software_idx].text.strip()
                creator_name = cols[creator_idx].text.strip()
                raw_initial_release = cols[initial_release_idx].text.strip()
                # Clean 'Initial release' string (remove bracketed references)
                clean_initial_release = re.sub(r'\[.*?\]', '', raw_initial_release).strip()
                cleaned_data_records.append({
                    'Software': software_name,
                    'Creator': creator_name,
                    'Initial_Release_Date': clean_initial_release
                })
            else:
                print(f"Skipping row {row_num + 1} due to insufficient columns or unexpected structure.")

        print("Cleaned and Standardized Data (First 5 records):")
        for record in cleaned_data_records[:5]:
            print(record)

        df_cleaned = pd.DataFrame(cleaned_data_records)
        print("\nDataFrame Info after Cleaning:")
        print(df_cleaned.info())
        print("\nDataFrame Head (cleaned data):")
        print(df_cleaned.head().to_string())

    else:
        print("Table with class 'wikitable' not found. Check URL or table class on the page.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")
finally:
    print("--- End of Procedure ---")

--- PROCEDURE 8: Clean and Standardize Extracted Data (Revised for Wikipedia Table) ---
Skipping row 31 due to insufficient columns or unexpected structure.
Cleaned and Standardized Data (First 5 records):
{'Software': 'BigDL', 'Creator': 'Jason Dai (Intel)', 'Initial_Release_Date': '2016'}
{'Software': 'Caffe', 'Creator': 'Berkeley Vision and Learning Center', 'Initial_Release_Date': '2013'}
{'Software': 'Chainer', 'Creator': 'Preferred Networks', 'Initial_Release_Date': '2015'}
{'Software': 'Deeplearning4j', 'Creator': 'Skymind engineering team; Deeplearning4j community; originally Adam Gibson', 'Initial_Release_Date': '2014'}
{'Software': 'DeepSpeed', 'Creator': 'Microsoft', 'Initial_Release_Date': '2019'}

DataFrame Info after Cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Software              30 non-null     str  
 1   Creato

In [23]:
#TRY IT TASK 8 - Revised for Wikipedia Table with Different Headers
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"
headers = {'User-Agent': 'Mozilla/5.0'}

print("--- PROCEDURE 8: Clean Numerical Data with Commas (Countries by Population) ---")

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    table = soup.find('table', class_='wikitable')

    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]

        print(f"Found headers: {headers_text}\n")

        # Define the indices for columns we want to extract
        country_idx = headers_text.index('Country') if 'Country' in headers_text else 0
        population_idx = headers_text.index('Population') if 'Population' in headers_text else -1

        # Check if required headers were found
        if population_idx == -1:
            print("ERROR: 'Population' column not found.")
            print("Found headers:", headers_text)
            exit()

        cleaned_data_records = []
        data_rows = table.find_all('tr')[1:]
        # Skip header row

        # Process all rows in the table
        for row_num, row in enumerate(data_rows):
            cols = row.find_all('td')
            # Ensure row has enough columns
            if len(cols) > max(country_idx, population_idx):
                country_name = cols[country_idx].text.strip()
                raw_population = cols[population_idx].text.strip()

                # Clean the population string: remove commas and brackets
                clean_population = re.sub(r'\[.*?\]', '', raw_population).strip()
                clean_population = re.sub(r',', '', clean_population).strip()

                # Try to convert to integer
                try:
                    population_int = int(clean_population)
                    cleaned_data_records.append({
                        'Country': country_name,
                        'Population_Raw': raw_population,
                        'Population_Cleaned': clean_population,
                        'Population_Integer': population_int
                    })
                except ValueError:
                    # Skip rows that can't be converted to integer
                    print(f"Skipping row {row_num + 1} - Could not convert '{clean_population}' to integer")
                    continue
            else:
                print(f"Skipping row {row_num + 1} due to insufficient columns.")

        print("Cleaned and Standardized Data (First 10 records):")
        for i, record in enumerate(cleaned_data_records[:10]):
            print(f"{i+1}. Country: {record['Country']}")
            print(f"   Raw: {record['Population_Raw']}")
            print(f"   Cleaned: {record['Population_Cleaned']}")
            print(f"   Integer: {record['Population_Integer']:,}\n")

        df_cleaned = pd.DataFrame(cleaned_data_records)
        print("\nDataFrame Info after Cleaning:")
        print(df_cleaned.info())
        print("\nDataFrame Head (cleaned data):")
        print(df_cleaned.head(10).to_string())
        print(f"\nTotal countries extracted: {len(df_cleaned)}")

    else:
        print("Table with class 'wikitable' not found. Check URL or table class on the page.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")
finally:
    print("--- End of Procedure ---")

--- PROCEDURE 8: Clean Numerical Data with Commas (Countries by Population) ---
Found headers: ['Location', 'Population', '% ofworld', 'Date', 'Source (official or fromthe United Nations)', 'Notes']

Cleaned and Standardized Data (First 10 records):
1. Country: World
   Raw: 8,232,000,000
   Cleaned: 8232000000
   Integer: 8,232,000,000

2. Country: India
   Raw: 1,417,492,000
   Cleaned: 1417492000
   Integer: 1,417,492,000

3. Country: China
   Raw: 1,404,890,000
   Cleaned: 1404890000
   Integer: 1,404,890,000

4. Country: United States
   Raw: 341,784,857
   Cleaned: 341784857
   Integer: 341,784,857

5. Country: Indonesia
   Raw: 288,315,089
   Cleaned: 288315089
   Integer: 288,315,089

6. Country: Pakistan
   Raw: 241,499,431
   Cleaned: 241499431
   Integer: 241,499,431

7. Country: Nigeria
   Raw: 223,800,000
   Cleaned: 223800000
   Integer: 223,800,000

8. Country: Brazil
   Raw: 213,421,037
   Cleaned: 213421037
   Integer: 213,421,037

9. Country: Bangladesh
   Raw: 169,82

In [28]:
import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 9: Implement Error Handling and Robustness (Modified) ---")

# Test URLs designed to trigger different error types
test_scenarios = {
    "valid_article_page": "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article": "https://www.example.com/this-page-does-not-exist",
    # Definitely returns 404
    "slow_server_sim_high_timeout": "http://httpbin.org/delay/3",
    # Will succeed with timeout=30
    "slow_server_sim_low_timeout": "http://httpbin.org/delay/3",
    # Will timeout with timeout=1
    "invalid_domain_connect": "http://this.domain.does.not.resolve.abc"
    # Designed for connection error
}

# Define different timeout values for specific scenarios
timeout_values = {
    "valid_article_page": 10,
    "non_existent_article": 10,
    "slow_server_sim_high_timeout": 30,
    # High timeout - should succeed
    "slow_server_sim_low_timeout": 1,
    # Low timeout - should fail
    "invalid_domain_connect": 5
}

for scenario_name, test_url in test_scenarios.items():
    timeout = timeout_values.get(scenario_name, 5)
    print(f"\nTesting Scenario: '{scenario_name}' at {test_url}")
    print(f"Timeout set to: {timeout} seconds")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        # Crucial: Set a reasonable timeout to prevent infinite hangs
        response = requests.get(test_url, headers=headers, timeout=timeout)
        # Raise an exception for HTTP error codes (4xx, 5xx)
        response.raise_for_status()
        # If successful, attempt to parse and extract a key element
        soup = BeautifulSoup(response.text, 'lxml')
        # Using 'lxml' parser
        # Selector for Wikipedia article title (h1 with id='firstHeading')
        if scenario_name == "valid_article_page":
            main_title_element = soup.find('h1', id='firstHeading')
        else:
            # For other scenarios, a generic h1 might be fine or it will fail as expected
            main_title_element = soup.find('h1')
        if main_title_element:
            print(f"SUCCESS: Fetched and found title: '{main_title_element.text.strip()[:100]}...'")
        else:
            print("SUCCESS (with caveat): Fetched page, but main title element not found.")

    except requests.exceptions.HTTPError as e:
        print(f"ERROR (HTTP): Status Code {e.response.status_code} for {test_url}. Reason: {e.response.reason}")
    except requests.exceptions.ConnectionError as e:
        print(f"ERROR (Connection): Could not establish connection to {test_url}. Details: {e}")
    except requests.exceptions.Timeout as e:
        print(f"ERROR (Timeout): Request to {test_url} timed out after {timeout} seconds. Details: {e}")
    except requests.exceptions.RequestException as e:
        print(f"ERROR (General Request): An unexpected request error occurred for {test_url}. Details: {e}")
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {test_url}. Details: {e}")

print("---")

--- PROCEDURE 9: Implement Error Handling and Robustness (Modified) ---

Testing Scenario: 'valid_article_page' at https://en.wikipedia.org/wiki/Artificial_intelligence
Timeout set to: 10 seconds
SUCCESS: Fetched and found title: 'Artificial intelligence...'

Testing Scenario: 'non_existent_article' at https://www.example.com/this-page-does-not-exist
Timeout set to: 10 seconds
ERROR (HTTP): Status Code 404 for https://www.example.com/this-page-does-not-exist. Reason: Not Found

Testing Scenario: 'slow_server_sim_high_timeout' at http://httpbin.org/delay/3
Timeout set to: 30 seconds
ERROR (Connection): Could not establish connection to http://httpbin.org/delay/3. Details: HTTPConnectionPool(host='httpbin.org', port=80): Max retries exceeded with url: /delay/3 (Caused by NameResolutionError("HTTPConnection(host='httpbin.org', port=80): Failed to resolve 'httpbin.org' ([Errno 11001] getaddrinfo failed)"))

Testing Scenario: 'slow_server_sim_low_timeout' at http://httpbin.org/delay/3
Timeo

In [27]:
#TRY IT TASK 9
import requests
from bs4 import BeautifulSoup

print("--- PROCEDURE 9: Implement Error Handling and Robustness (Modified) ---")

# Test URLs designed to trigger different error types
test_scenarios = {
    "valid_article_page": "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "non_existent_article": "https://www.example.com/this-page-does-not-exist",
    # Definitely returns 404
    "slow_server_sim_high_timeout": "http://httpbin.org/delay/3",
    # Will succeed with timeout=30
    "slow_server_sim_low_timeout": "http://httpbin.org/delay/3",
    # Will timeout with timeout=1
    "invalid_domain_connect": "http://this.domain.does.not.resolve.abc"
    # Designed for connection error
}

# Define different timeout values for specific scenarios
timeout_values = {
    "valid_article_page": 10,
    "non_existent_article": 10,
    "slow_server_sim_high_timeout": 30,
    # High timeout - should succeed
    "slow_server_sim_low_timeout": 1,
    # Low timeout - should fail
    "invalid_domain_connect": 5
}

for scenario_name, test_url in test_scenarios.items():
    timeout = timeout_values.get(scenario_name, 5)
    print(f"\nTesting Scenario: '{scenario_name}' at {test_url}")
    print(f"Timeout set to: {timeout} seconds")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        # Crucial: Set a reasonable timeout to prevent infinite hangs
        response = requests.get(test_url, headers=headers, timeout=timeout)
        # Raise an exception for HTTP error codes (4xx, 5xx)
        response.raise_for_status()
        # If successful, attempt to parse and extract a key element
        soup = BeautifulSoup(response.text, 'lxml')
        # Using 'lxml' parser
        # Selector for Wikipedia article title (h1 with id='firstHeading')
        if scenario_name == "valid_article_page":
            main_title_element = soup.find('h1', id='firstHeading')
        else:
            # For other scenarios, a generic h1 might be fine or it will fail as expected
            main_title_element = soup.find('h1')
        if main_title_element:
            print(f"SUCCESS: Fetched and found title: '{main_title_element.text.strip()[:100]}...'")
        else:
            print("SUCCESS (with caveat): Fetched page, but main title element not found.")

    except requests.exceptions.HTTPError as e:
        print(f"ERROR (HTTP): Status Code {e.response.status_code} for {test_url}. Reason: {e.response.reason}")
    except requests.exceptions.ConnectionError as e:
        print(f"ERROR (Connection): Could not establish connection to {test_url}. Details: {e}")
    except requests.exceptions.Timeout as e:
        print(f"ERROR (Timeout): Request to {test_url} timed out after {timeout} seconds. Details: {e}")
    except requests.exceptions.RequestException as e:
        print(f"ERROR (General Request): An unexpected request error occurred for {test_url}. Details: {e}")
    except Exception as e:
        print(f"ERROR (Unexpected): An unhandled error occurred for {test_url}. Details: {e}")

print("---")

--- PROCEDURE 9: Implement Error Handling and Robustness (Modified) ---

Testing Scenario: 'valid_article_page' at https://en.wikipedia.org/wiki/Artificial_intelligence
Timeout set to: 10 seconds
SUCCESS: Fetched and found title: 'Artificial intelligence...'

Testing Scenario: 'non_existent_article' at https://www.example.com/this-page-does-not-exist
Timeout set to: 10 seconds
ERROR (HTTP): Status Code 404 for https://www.example.com/this-page-does-not-exist. Reason: Not Found

Testing Scenario: 'slow_server_sim_high_timeout' at http://httpbin.org/delay/3
Timeout set to: 30 seconds
ERROR (Connection): Could not establish connection to http://httpbin.org/delay/3. Details: HTTPConnectionPool(host='httpbin.org', port=80): Max retries exceeded with url: /delay/3 (Caused by NameResolutionError("HTTPConnection(host='httpbin.org', port=80): Failed to resolve 'httpbin.org' ([Errno 11001] getaddrinfo failed)"))

Testing Scenario: 'slow_server_sim_low_timeout' at http://httpbin.org/delay/3
Timeo

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os  # For verifying file operations
import re  # For cleaning headers
import sqlite3  # Import for SQLite database operations

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

# Step 1: Re-collect some sample data from the deep learning software comparison table
# This step builds on PROCEDURE 1 (Fetching), PROCEDURE 2 (Parsing), and PROCEDURE 3 (Basic Extraction)

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}  # Standard User-Agent header for requests

try:
    response = requests.get(url, headers=headers, timeout=10)  # Added timeout for robustness (from PROCEDURE 9)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')  # Using 'lxml' parser
    
    final_data_to_save = []
    table = soup.find('table', class_='wikitable')  # Locate the main table
    
    if table:
        # Extract raw headers from the first row of the table
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]

        # Standardize headers for easier use in Pandas DataFrame columns.
        # This removes bracketed references (like [a], [1]), parentheses content,
        # and replaces spaces with underscores. (From PROCEDURE 8: Cleaning)
        table_headers = [re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip() for h in table_headers_raw]
        
        data_rows = table.find_all('tr')[1:]  # Get all data rows, skipping the header row
        
        # Iterate through data rows to extract content
        # Collecting up to 15 rows for demonstration, adjust as needed.
        for i, row in enumerate(data_rows[:15]):
            cols = row.find_all('td')  # Get all data cells in the current row

            row_data = {}
            # Map column values to their respective standardized headers based on position
            for j, col in enumerate(cols):
                if j < len(table_headers):  # Ensure we don't go out of bounds of our defined headers
                    header_key = table_headers[j]
                    row_data[header_key] = col.text.strip()  # Extract text content and strip whitespace

            if row_data:  # Only add the row if it successfully extracted some data
                final_data_to_save.append(row_data)
    else:
        print("ERROR: Table with class 'wikitable' not found on the page. Check URL or table class.")
    
    if final_data_to_save:
        df_output = pd.DataFrame(final_data_to_save)
        print("\n--- Data Ready for Archiving (DataFrame Head) ---")
        print(df_output.head().to_string())  # Display the first few rows of the DataFrame
        
        # Step 2: Store data as a CSV file (common for tabular data)
        csv_file_path = 'deep_learning_software_comparison.csv'
        df_output.to_csv(csv_file_path, index=False, encoding='utf-8')  # index=False prevents writing DataFrame index as a column
        print(f"\nData successfully saved to: {csv_file_path} (CSV format)")
        
        # Step 3: Store data as a JSON file (good for hierarchical data or API-like output)
        json_file_path = 'deep_learning_software_comparison.json'
        # 'orient='records'' saves as a list of dictionaries, which is a common and readable JSON format.
        # 'indent=4' makes the JSON output human-readable with indentation.
        df_output.to_json(json_file_path, orient='records', indent=4)
        print(f"Data successfully saved to: {json_file_path} (JSON format)")
        
        # Step 4: Store data in an SQLite database
        db_file_path = 'deep_learning_software_comparison.db'
        table_name = 'dl_software_comparison'  # Name of the table within the SQLite database
        conn = None  # Initialize connection to None
        
        try:
            conn = sqlite3.connect(db_file_path)  # Establish connection to the SQLite database file
            # Use Pandas to_sql to write the DataFrame directly to an SQLite table.
            # if_exists='replace' will drop the table if it already exists and create a new one.
            # Other options: 'append' (add new rows) or 'fail' (raise error if table exists).
            # index=False prevents Pandas from writing the DataFrame's internal index as a column in the DB.
            df_output.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"Data successfully saved to: {db_file_path} (SQLite database, table: '{table_name}')")
        except Exception as db_err:
            print(f"ERROR (SQLite): Could not save data to SQLite: {db_err}")
        finally:
            if conn:  # Ensure the connection object exists before trying to close it
                conn.close()  # Always close the database connection
        
        # Step 5: Verify all files were created
        # Now check for CSV, JSON, and SQLite database files
        all_files_exist = os.path.exists(csv_file_path) and \
                          os.path.exists(json_file_path) and \
                          os.path.exists(db_file_path)
        
        if all_files_exist:
            print("\nVerification: CSV, JSON, and SQLite files are all present in the directory.")
        else:
            print("\nVerification Failed: One or more files were not found after saving.")
    else:
        print("No data collected to proceed with storage.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")


--- PROCEDURE 10: Store the Extracted Data ---

--- Data Ready for Archiving (DataFrame Head) ---
         Software                                                                     Creator Initial_release Software_license Open_source                                         Platform         Written_in                                     Interface OpenMP_support        OpenCL_support CUDA_support ROCm_support Automatic_differentiation Has_pretrained_models Recurrent_nets Convolutional_nets RBM/DBNs Parallel_execution(multi_node) Actively_developed
0           BigDL                                                           Jason Dai (Intel)            2016       Apache 2.0         Yes                                     Apache Spark              Scala                                 Scala, Python                                                No           No                                             Yes            Yes                Yes                                               

In [4]:
#TRY IT TASK 10 - Revised for Wikipedia Table with Different Headers
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os  # For verifying file operations
import re  # For cleaning headers
import sqlite3  # Import for SQLite database operations

print("\n--- PROCEDURE 10: Store the Extracted Data ---")

# Step 1: Re-collect some sample data from the deep learning software comparison table
# This step builds on PROCEDURE 1 (Fetching), PROCEDURE 2 (Parsing), and PROCEDURE 3 (Basic Extraction)

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}  # Standard User-Agent header for requests

try:
    response = requests.get(url, headers=headers, timeout=10)  # Added timeout for robustness (from PROCEDURE 9)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')  # Using 'lxml' parser
    
    final_data_to_save = []
    table = soup.find('table', class_='wikitable')  # Locate the main table
    
    if table:
        # Extract raw headers from the first row of the table
        table_headers_raw = [th.text.strip() for th in table.find('tr').find_all('th')]

        # Standardize headers for easier use in Pandas DataFrame columns.
        # This removes bracketed references (like [a], [1]), parentheses content,
        # and replaces spaces with underscores. (From PROCEDURE 8: Cleaning)
        table_headers = [re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip() for h in table_headers_raw]
        
        data_rows = table.find_all('tr')[1:]  # Get all data rows, skipping the header row
        
        # Iterate through data rows to extract content
        # Collecting up to 15 rows for demonstration, adjust as needed.
        for i, row in enumerate(data_rows[:15]):
            cols = row.find_all('td')  # Get all data cells in the current row

            row_data = {}
            # Map column values to their respective standardized headers based on position
            for j, col in enumerate(cols):
                if j < len(table_headers):  # Ensure we don't go out of bounds of our defined headers
                    header_key = table_headers[j]
                    row_data[header_key] = col.text.strip()  # Extract text content and strip whitespace

            if row_data:  # Only add the row if it successfully extracted some data
                final_data_to_save.append(row_data)
    else:
        print("ERROR: Table with class 'wikitable' not found on the page. Check URL or table class.")
    
    if final_data_to_save:
        df_output = pd.DataFrame(final_data_to_save)
        print("\n--- Data Ready for Archiving (DataFrame Head) ---")
        print(df_output.head().to_string())  # Display the first few rows of the DataFrame
        
        # Step 2: Store data as a CSV file (common for tabular data)
        csv_file_path = 'deep_learning_software_comparison.csv'
        df_output.to_csv(csv_file_path, index=False, encoding='utf-8')  # index=False prevents writing DataFrame index as a column
        print(f"\nData successfully saved to: {csv_file_path} (CSV format)")
        
        # Step 3: Store data as a JSON file (good for hierarchical data or API-like output)
        json_file_path = 'deep_learning_software_comparison.json'
        # 'orient='records'' saves as a list of dictionaries, which is a common and readable JSON format.
        # 'indent=4' makes the JSON output human-readable with indentation.
        df_output.to_json(json_file_path, orient='records', indent=4)
        print(f"Data successfully saved to: {json_file_path} (JSON format)")
        
        # Step 4: Store data in an SQLite database
        db_file_path = 'deep_learning_software_comparison.db'
        table_name = 'dl_software_comparison'  # Name of the table within the SQLite database
        conn = None  # Initialize connection to None
        
        try:
            conn = sqlite3.connect(db_file_path)  # Establish connection to the SQLite database file
            # Use Pandas to_sql to write the DataFrame directly to an SQLite table.
            # if_exists='replace' will drop the table if it already exists and create a new one.
            # Other options: 'append' (add new rows) or 'fail' (raise error if table exists).
            # index=False prevents Pandas from writing the DataFrame's internal index as a column in the DB.
            df_output.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"Data successfully saved to: {db_file_path} (SQLite database, table: '{table_name}')")
        except Exception as db_err:
            print(f"ERROR (SQLite): Could not save data to SQLite: {db_err}")
        finally:
            if conn:  # Ensure the connection object exists before trying to close it
                conn.close()  # Always close the database connection
        
        # Step 5: Verify all files were created
        # Now check for CSV, JSON, and SQLite database files
        all_files_exist = os.path.exists(csv_file_path) and \
                          os.path.exists(json_file_path) and \
                          os.path.exists(db_file_path)
        
        if all_files_exist:
            print("\nVerification: CSV, JSON, and SQLite files are all present in the directory.")
        else:
            print("\nVerification Failed: One or more files were not found after saving.")
    else:
        print("No data collected to proceed with storage.")

except requests.exceptions.RequestException as e:
    print(f"ERROR (Network/HTTP): Could not fetch {url}: {e}")
except Exception as e:
    print(f"ERROR (Unexpected): An unhandled error occurred: {e}")


--- PROCEDURE 10: Store the Extracted Data ---

--- Data Ready for Archiving (DataFrame Head) ---
         Software                                                                     Creator Initial_release Software_license Open_source                                         Platform         Written_in                                     Interface OpenMP_support        OpenCL_support CUDA_support ROCm_support Automatic_differentiation Has_pretrained_models Recurrent_nets Convolutional_nets RBM/DBNs Parallel_execution(multi_node) Actively_developed
0           BigDL                                                           Jason Dai (Intel)            2016       Apache 2.0         Yes                                     Apache Spark              Scala                                 Scala, Python                                                No           No                                             Yes            Yes                Yes                                               

In [5]:
# Fetching HTML Status

import requests

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    print(f"HTTP Status Code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")

HTTP Status Code: 200


In [7]:
# Parsing HTML and Getting Title Tag

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    print(f"Page Title (from <title> tag): {soup.title.string}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except AttributeError:
    print("Title tag not found on the page.")

Page Title (from <title> tag): Web scraping - Wikipedia


In [9]:
# Extracting Main Article Heading

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    # The main heading of the Wikipedia article
    main_heading = soup.find('h1', id='firstHeading')
    
    if main_heading:
        print(f"Main Heading: '{main_heading.text.strip()}'")
    else:
        print("Main heading (h1 with id='firstHeading') not found.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Main Heading: 'Artificial intelligence'


In [11]:
# Extracting First Paragraph

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    content_div = soup.find('div', id='mw-contenttext')
    first_paragraph = None
    
    if content_div:
        paragraphs = content_div.find_all('p')
        for p in paragraphs:
            text = p.text.strip()
            # Skip paragraphs inside infoboxes and empty paragraphs
            if text and not p.find_parent('table', class_='infobox'):
                first_paragraph = text
                break
    
    if first_paragraph:
        # A truncated portion of the first significant paragraph from the article content
        print(f"First Paragraph (truncated): '{first_paragraph[:150]}...'")
    else:
        print("First meaningful paragraph not found.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

First meaningful paragraph not found.


In [12]:
# Extracting the first quote and author from Quotes To Scrape

import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"  # Changed URL to a known scraping practice site
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')
    
    # Find the first div that represents a quote
    first_quote_div = soup.find('div', class_='quote')
    
    if first_quote_div:
        # Extract the quote text
        quote_text_element = first_quote_div.find('span', class_='text')
        # Extract the author's name
        author_element = first_quote_div.find('small', class_='author')
        
        if quote_text_element and author_element:
            quote = quote_text_element.text.strip()
            author = author_element.text.strip()
            # The text of the first quote and the name of its author, as displayed on the page
            print(f"Sample Quote: '{quote}'")
            print(f"Sample Author: {author}")
        else:
            print("No usable quote text or author element found within the first quote div.")
    else:
        print("No quote div found. The website structure may have changed, or a different selector is needed.")
        
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Sample Quote: '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'
Sample Author: Albert Einstein


In [13]:
# Extracting Quote Author from Quotes To Scrape

import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')
    
    # Step 1: Find the first div that contains a quote (which also contains the author)
    first_quote_div = soup.find('div', class_='quote')
    
    if first_quote_div:
        # Step 2: Within this quote div, find the author's name element
        author_element = first_quote_div.find('small', class_='author')
        
        if author_element:
            author_name = author_element.text.strip()
            # The name of the author of the first quote found on the page
            print(f"Author of the first quote: '{author_name}'")
        else:
            print("Author element not found within the first quote div (class='author').")
    else:
        print("First quote container (div with class 'quote') not found. The website structure may have changed, or a different selector is needed.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Author of the first quote: 'Albert Einstein'


In [15]:
# Extracting ALL Quote Authors from Quotes To Scrape

import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')
    
    # Find ALL div elements that contain a quote
    all_quote_divs = soup.find_all('div', class_='quote')
    
    if all_quote_divs:
        print("Authors found on the page:")
        authors_list = []
        
        for i, quote_div in enumerate(all_quote_divs):
            # Within each quote div, find the author's name element
            author_element = quote_div.find('small', class_='author')
            
            if author_element:
                author_name = author_element.text.strip()
                authors_list.append(author_name)
                print(f"- {author_name}")
            else:
                print(f"- Author not found for quote #{i+1}.")
        
        # A list of all author names found on the page, each preceded by a hyphen,
        # followed by the total count of unique authors found on that page
        print(f"\nTotal unique authors found: {len(set(authors_list))}")  # Using set to count unique authors
    else:
        print("No quote containers (divs with class 'quote') found on the page.")
        
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Authors found on the page:
- Albert Einstein
- J.K. Rowling
- Albert Einstein
- Jane Austen
- Marilyn Monroe
- Albert Einstein
- André Gide
- Thomas A. Edison
- Eleanor Roosevelt
- Steve Martin

Total unique authors found: 8


In [17]:
# Raw Numerical String with Commas

import requests
from bs4 import BeautifulSoup
import re  # Import re for potential future cleaning if needed

url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, 'lxml')
    
    table = soup.find('table', class_='wikitable')
    
    # The raw numerical string representing the population of the first entry
    # (typically "World") from the main population table on the Wikipedia page
    
    if table:
        header_cells = table.find('tr').find_all('th')
        headers_text = [th.text.strip() for th in header_cells]
        
        population_idx = -1
        for i, header in enumerate(headers_text):
            if 'Population' in header:
                population_idx = i
                break
        
        if population_idx != -1:
            first_data_row = table.find_all('tr')[1]
            cols = first_data_row.find_all('td')
            
            if len(cols) > population_idx:
                raw_population = cols[population_idx].text.strip()
                print(f"Raw Population String (e.g., World population): '{raw_population}'")
            else:
                print("Population column not found in the first data row (index out of bounds).")
        else:
            print("Population header not found in table. Check header text for variations (e.g., 'Population (July 2024 est.)').")
    else:
        print("Table with class 'wikitable' not found on the page.")
        
except requests.exceptions.RequestException as e:
    print(f"Error fetching the page: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Raw Population String (e.g., World population): '8,232,000,000'


In [18]:
# Save Laptops Data to CSV

import pandas as pd

items = soup.select(".thumbnail")
data = []

for it in items:
    data.append({
        "name": it.select_one(".title")["title"],
        "price": it.select_one(".price").text
    })

df = pd.DataFrame(data)
df.to_csv("laptops.csv", index=False)
print("Saved laptops.csv")

Saved laptops.csv


In [19]:
# Handling Timeout Error

import requests

url = "http://httpbin.org/delay/6"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=5)
    response.raise_for_status()
    print("SUCCESS: Page fetched (unexpected for timeout).")
except requests.exceptions.Timeout as e:
    print(f"ERROR (Timeout): Request to {url} timed out after 5 seconds. Details: {e}")
except requests.exceptions.RequestException as e:
    print(f"ERROR (General Request): {e}")

ERROR (General Request): HTTPConnectionPool(host='httpbin.org', port=80): Max retries exceeded with url: /delay/6 (Caused by NameResolutionError("HTTPConnection(host='httpbin.org', port=80): Failed to resolve 'httpbin.org' ([Errno 11001] getaddrinfo failed)"))


In [20]:
# Storing Data to CSV

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

scraped_data = []

# Confirmation that a CSV file containing the DataFrame's data has been successfully created and saved to disk

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')
    
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    
    scraped_data.append({
        'Quote': quote,
        'Author': author
    })

df_scraped = pd.DataFrame(scraped_data)
csv_file_path = 'web_scraped_quotes.csv'
df_scraped.to_csv(csv_file_path, index=False, encoding='utf-8')
print(f"CSV file saved: {csv_file_path}")


CSV file saved: web_scraped_quotes.csv


In [21]:
# Storing Web Scraped Data to JSON

import requests
from bs4 import BeautifulSoup
import json  # Import the json module
import os

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

# Confirmation that a JSON file representing the DataFrame's data has been successfully created and saved to disk

scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')
    
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    
    scraped_data.append({
        'quote': quote,  # Use consistent lowercase keys for JSON
        'author': author
    })

json_file_path = 'web_scraped_quotes.json'

# Store the scraped data as JSON
with open(json_file_path, 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)  # indent for pretty printing

print(f"JSON file saved: {json_file_path}")

JSON file saved: web_scraped_quotes.json


In [22]:
# Storing Web Scraped Data to SQLite

import requests
from bs4 import BeautifulSoup
import sqlite3  # Import the sqlite3 module
import os

url = "http://quotes.toscrape.com/"

# Confirmation that the DataFrame's data has been successfully written to a specified table within an SQLite database file

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

scraped_data = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

all_quote_divs = soup.find_all('div', class_='quote')

for quote_div in all_quote_divs:
    quote_text_element = quote_div.find('span', class_='text')
    author_element = quote_div.find('small', class_='author')
    
    quote = quote_text_element.text.strip() if quote_text_element else "N/A"
    author = author_element.text.strip() if author_element else "N/A"
    
    scraped_data.append({
        'quote': quote,
        'author': author
    })

# --- Storing Data to SQLite ---

db_file_path = 'web_scraped_quotes.db'  # Define the SQLite database file name
conn = sqlite3.connect(db_file_path)  # Connect to the SQLite database
cursor = conn.cursor()  # Create a cursor object

# Create table if it doesn't exist
# Using a primary key for efficient data management
cursor.execute('''
    CREATE TABLE IF NOT EXISTS quotes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        quote TEXT NOT NULL,
        author TEXT NOT NULL
    )
''')

# Insert scraped data into the table
for item in scraped_data:
    cursor.execute("INSERT INTO quotes (quote, author) VALUES (?, ?)", (item['quote'], item['author']))

conn.commit()  # Commit changes to the database
conn.close()  # Close the connection

print(f"SQLite database saved: {db_file_path}")

SQLite database saved: web_scraped_quotes.db


In [1]:
import pandas as pd
import sqlite3
import os

db_file_path = 'web_scraped_quotes.db'
table_name = 'quotes'

# Check if the database file exists
if os.path.exists(db_file_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_file_path)
    
    # Read data from the 'quotes' table into a pandas DataFrame
    df_read = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    
    # Print the data read from SQLite
    print("\nData read from SQLite ('web_scraped_quotes.db'):")
    print(df_read.to_string())
    
    # Close the connection
    conn.close()
else:
    print(f"SQLite database '{db_file_path}' not found. Please ensure the data was stored to SQLite first.")


Data read from SQLite ('web_scraped_quotes.db'):
   id                                                                                                                                quote             author
0   1                  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”    Albert Einstein
1   2                                                “It is our choices, Harry, that show what we truly are, far more than our abilities.”       J.K. Rowling
2   3  “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”    Albert Einstein
3   4                             “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”        Jane Austen
4   5                      “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”     Marilyn Monroe
5 

In [23]:
# Extracting Quotes from First Page

import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    quote_elements = soup.find_all('div', class_='quote')
    
    if quote_elements:
        print(f"Found {len(quote_elements)} quotes on the first page.")
        for i, quote_item in enumerate(quote_elements[:2]):
            text = quote_item.find('span', class_='text').text.strip()
            author = quote_item.find('small', class_='author').text.strip()
            print(f"Quote {i+1}: \"{text[:70]}...\" - {author}")
    else:
        print("No quotes found on the first page.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Found 10 quotes on the first page.
Quote 1: "“The world as we have created it is a process of our thinking. It cann..." - Albert Einstein
Quote 2: "“It is our choices, Harry, that show what we truly are, far more than ..." - J.K. Rowling


In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "http://quotes.toscrape.com/"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    # The full URL of the "next page" pagination link, essential for navigating multi-page content.
    next_li = soup.find('li', class_='next')
    
    if next_li:
        next_link_element = next_li.find('a', href=True)
        if next_link_element:
            next_page_relative_url = next_link_element['href']
            full_next_page_url = urljoin(url, next_page_relative_url)
            print(f"Found 'Next Page' link: {full_next_page_url}")
        else:
            print("Next page tag not found within 'next' li.")
    else:
        print("No 'next' pagination li found (might be on the last page).")

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Found 'Next Page' link: http://quotes.toscrape.com/page/2/


In [24]:
# Extracting Raw Table Headers

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    table = soup.find('table', class_='wikitable')
    
    if table:
        header_cells = table.find('tr').find_all('th')
        raw_headers = [th.text.strip() for th in header_cells]
        # A list of raw, unprocessed column headers extracted directly from an HTML table,
        # potentially containing special characters or references
        print(f"Raw Headers: {raw_headers}")
    else:
        print("Table not found.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Raw Headers: ['Software', 'Creator', 'Initial release', 'Software license[a]', 'Open source', 'Platform', 'Written in', 'Interface', 'OpenMP support', 'OpenCL support', 'CUDA support', 'ROCm support[1]', 'Automatic differentiation[2]', 'Has pretrained models', 'Recurrent nets', 'Convolutional nets', 'RBM/DBNs', 'Parallel execution(multi node)', 'Actively developed']


In [25]:
# Standardizing Table Headers

import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Comparison_of_deep_learning_software"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

raw_headers = []

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'lxml')

# Find the main table
table = soup.find('table', class_='wikitable')

# Find all table header cells (<th>) in the first row (<tr>)
header_cells = table.find('tr').find_all('th')
raw_headers = [th.text.strip() for th in header_cells]

print(f"Raw Headers from Web Scraping: {raw_headers}")

# The original standardization logic
standardized_headers = [re.sub(r'\[.*?\]|\s\(.*?\)', '', h).replace(' ', '_').strip() for h in raw_headers]

print(f"Standardized Headers: {standardized_headers}")

Raw Headers from Web Scraping: ['Software', 'Creator', 'Initial release', 'Software license[a]', 'Open source', 'Platform', 'Written in', 'Interface', 'OpenMP support', 'OpenCL support', 'CUDA support', 'ROCm support[1]', 'Automatic differentiation[2]', 'Has pretrained models', 'Recurrent nets', 'Convolutional nets', 'RBM/DBNs', 'Parallel execution(multi node)', 'Actively developed']
Standardized Headers: ['Software', 'Creator', 'Initial_release', 'Software_license', 'Open_source', 'Platform', 'Written_in', 'Interface', 'OpenMP_support', 'OpenCL_support', 'CUDA_support', 'ROCm_support', 'Automatic_differentiation', 'Has_pretrained_models', 'Recurrent_nets', 'Convolutional_nets', 'RBM/DBNs', 'Parallel_execution(multi_node)', 'Actively_developed']


In [28]:
ERROR (Connection): Could not establish connection
to http://this.domain.does.not.resolve.abc. Details:
HTTPConnectionPool(host='this.domain.does.not.resolv
e.abc', port=80): Max retries exceeded with url: /
(Caused by
NameResolutionError("<urllib3.connection.HTTPConnect
ion object at 0x00000207B2AA3DF0>: Failed to resolve
'this.domain.does.not.resolve.abc' ([Errno 11001]
getaddrinfo failed)"))

SyntaxError: unterminated string literal (detected at line 3) (3283395512.py, line 3)

In [29]:
# Extracting Wikipedia Last Modified Date

import requests
from bs4 import BeautifulSoup
import re

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    
    last_modified_element = soup.find('li', id='footer-info-lastmod')
    
    if last_modified_element:
        raw_text = last_modified_element.text.strip()
        date_match = re.search(r'on (\d{1,2} \w+ \d{4})', raw_text)
        
        if date_match:
            clean_date = date_match.group(1)
            print(f"Last Modified Date: '{clean_date}'")
        else:
            print(f"Last Modified Date (raw, unable to parse): '{raw_text}'")
    else:
        print("Last modified element not found.")
        
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

Last Modified Date: '4 May 2026'
